In [1]:
import os

file_path_train = os.path.join(os.getcwd(),"jsb_chorales","train")
file_path_test = os.path.join(os.getcwd(),"jsb_chorales","test")
file_path_valid = os.path.join(os.getcwd(),"jsb_chorales","valid")


In [2]:
files_train = sorted(os.listdir(file_path_train))
files_test = sorted(os.listdir(file_path_test))
files_valid = sorted(os.listdir(file_path_valid))



In [3]:
import pandas as pd

def load_chorales(filepaths):
    return [pd.read_csv(filepath).values.tolist() for filepath in filepaths]

train_chorales = load_chorales(f"{file_path_train}/{files_train[i]}" for i in range(len(files_train)))
valid_chorales = load_chorales(f"{file_path_valid}/{files_valid[i]}" for i in range(len(files_valid)))
test_chorales = load_chorales(f"{file_path_test}/{files_test[i]}" for i in range(len(files_test)))

In [4]:
train_chorales[0]

[[74, 70, 65, 58],
 [74, 70, 65, 58],
 [74, 70, 65, 58],
 [74, 70, 65, 58],
 [75, 70, 58, 55],
 [75, 70, 58, 55],
 [75, 70, 60, 55],
 [75, 70, 60, 55],
 [77, 69, 62, 50],
 [77, 69, 62, 50],
 [77, 69, 62, 50],
 [77, 69, 62, 50],
 [77, 70, 62, 55],
 [77, 70, 62, 55],
 [77, 69, 62, 55],
 [77, 69, 62, 55],
 [75, 67, 63, 48],
 [75, 67, 63, 48],
 [75, 69, 63, 48],
 [75, 69, 63, 48],
 [74, 70, 65, 46],
 [74, 70, 65, 46],
 [74, 70, 65, 46],
 [74, 70, 65, 46],
 [72, 69, 65, 53],
 [72, 69, 65, 53],
 [72, 69, 65, 53],
 [72, 69, 65, 53],
 [72, 69, 65, 53],
 [72, 69, 65, 53],
 [72, 69, 65, 53],
 [72, 69, 65, 53],
 [74, 70, 65, 46],
 [74, 70, 65, 46],
 [74, 70, 65, 46],
 [74, 70, 65, 46],
 [75, 69, 63, 48],
 [75, 69, 63, 48],
 [75, 67, 63, 48],
 [75, 67, 63, 48],
 [77, 65, 62, 50],
 [77, 65, 62, 50],
 [77, 65, 60, 50],
 [77, 65, 60, 50],
 [74, 67, 58, 55],
 [74, 67, 58, 55],
 [74, 67, 58, 53],
 [74, 67, 58, 53],
 [72, 67, 58, 51],
 [72, 67, 58, 51],
 [72, 67, 58, 51],
 [72, 67, 58, 51],
 [72, 65, 57

In [5]:
notes = set()
for chorales in (train_chorales,valid_chorales,test_chorales):
    for chorale in chorales:
        for chord in chorale:
            notes |= set(chord)

n_notes = len(notes)
min_note = min(notes - {0})
max_note = max(notes)



In [6]:
n_notes

47

In [7]:
import numpy as np
from IPython.display import Audio

def note_to_frequencies(notes):
    return 2 ** ((np.array(notes)-69)/12)*440

def frequencies_to_samples(frequencies,tempo,sample_rate):
    note_duration = 60 / tempo
    frequencies = np.round(note_duration + frequencies) / note_duration
    n_samples = int(note_duration * sample_rate)
    time = np.linspace(0,note_duration,n_samples)
    sine_waves = np.sin(2 * np.pi * frequencies.reshape(-1,1) * time)
    sine_waves *= (frequencies > 9).reshape(-1,1)
    return sine_waves.reshape(-1)

def chords_to_samples(chords,tempo,sample_rate):
    freqs = note_to_frequencies(chords)
    freqs = np.r_[freqs,freqs[-1:]]
    merged = np.mean([frequencies_to_samples(melody,tempo,sample_rate) for melody in freqs.T],axis=0)
    n_fade_out_samples = sample_rate * 60 //tempo
    fade_out = np.linspace(1.,0.,n_fade_out_samples) ** 2
    merged[-n_fade_out_samples:] *= fade_out
    return merged


def play_chords(chords,tempo=160,amplitude=0.1,sample_rate=44100,filepath = None):
    samples = amplitude * chords_to_samples(chords,tempo,sample_rate)
    if filepath:
        from scipy.io import wavfile
        samples = (2**15 * samples).astype(np.int16)
        wavfile.write(filepath,sample_rate,samples)
        return display(Audio(filepath))
    else:
        return display(Audio(samples,rate=sample_rate))

In [8]:
import tensorflow as tf
def create_target(batch):
    x = batch[:,:-1]
    y = batch[:,1:]
    return x, y

def preprocess(window):
    window = tf.where(window == 0,window,window - min_note + 1)
    return tf.reshape(window,[-1])

def bach_dataset(chorales,batch_size=32,shuffle_buffer_size=None,window_size=32,window_shift=16,cache=True):
    
    def batch_window(window):
        return window.batch(window_size  + 1)
    
    def to_windows(chorale):
        dataset = tf.data.Dataset.from_tensor_slices(chorale)
        dataset = dataset.window(window_size + 1,shift=window_shift,drop_remainder=True)
        return dataset.flat_map(batch_window)
    
    chorales = tf.ragged.constant(chorales,ragged_rank=1)
    dataset = tf.data.Dataset.from_tensor_slices(chorales)
    dataset = dataset.flat_map(to_windows).map(preprocess)
    if cache:
        dataset = dataset.cache()

    if shuffle_buffer_size:
        dataset = dataset.shuffle(shuffle_buffer_size)
    
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(create_target)
    return dataset.prefetch(1)

2025-05-01 17:12:53.182395: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-01 17:12:53.308184: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746130373.352840    4782 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746130373.364963    4782 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-01 17:12:53.480628: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [9]:
train_set = bach_dataset(train_chorales,shuffle_buffer_size=1000)
valid_set =bach_dataset(valid_chorales)
test_set = bach_dataset(test_chorales)

I0000 00:00:1746130376.176487    4782 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1819 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2050, pci bus id: 0000:01:00.0, compute capability: 8.6


In [10]:
n_embedding_dims =5
model = tf.keras.models.Sequential([
    tf.keras.layers.Embedding(input_dim=n_notes,output_dim=n_embedding_dims,input_shape=[None]),
    tf.keras.layers.Conv1D(32,kernel_size=2,padding="causal",activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv1D(48,kernel_size=2,padding="causal",activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv1D(64,kernel_size=2,padding="causal",activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv1D(96,kernel_size=2,padding="causal",activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.LSTM(256,return_sequences=True),
    tf.keras.layers.Dense(n_notes,activation="softmax")
])

model.summary()

/home/lucas/Área de Trabalho/curso_tensorflow/venv/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, None, 5)        │           235 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, None, 32)       │           352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, None, 32)       │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, None, 48)       │         3,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, None, 48)       │           192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, None, 64)       │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, None, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, None, 96)       │        12,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, None, 96)       │           384 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, None, 256)      │       361,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, None, 47)       │        12,079 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 396,810 (1.51 MB)

 Trainable params: 396,330 (1.51 MB)

 Non-trainable params: 480 (1.88 KB)

In [11]:
optimizer = tf.keras.optimizers.Nadam(learning_rate=1e-3)

model.compile(loss="sparse_categorical_crossentropy",optimizer=optimizer,metrics=["accuracy"])
model.fit(train_set,epochs=30,validation_data=valid_set)

Epoch 1/30


I0000 00:00:1746130380.020192    6897 cuda_dnn.cc:529] Loaded cuDNN version 90800
2025-05-01 17:13:01.093692: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:306] Allocator (GPU_0_bfc) ran out of memory trying to allocate 3.04GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.


     98/Unknown 6s 23ms/step - accuracy: 0.4003 - loss: 2.5111

2025-05-01 17:13:03.605420: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2025-05-01 17:13:03.605451: I tensorflow/core/framework/local_rendezvous.cc:428] Local rendezvous send item cancelled. Key hash: 17048839311893398996
2025-05-01 17:13:03.605454: I tensorflow/core/framework/local_rendezvous.cc:428] Local rendezvous send item cancelled. Key hash: 3976896148490793123
2025-05-01 17:13:03.605457: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_2]]
2025-05-01 17:13:03.605528: I tensorflow/core/framework/local_rendezvous.cc:428] Local rendezvous send item cancelled. Key hash: 8205433285561994663
/home/lucas/Área de Trabalho/curso_tensorflow/venv/lib/python3.10/site-packages/keras/src/trainers/epoch_iterator.py:151: UserWarning: Your input ran out of da

98/98 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step - accuracy: 0.4022 - loss: 2.5033 - val_accuracy: 0.0699 - val_loss: 3.5565
Epoch 2/30
10/98 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.7514 - loss: 0.9872

...

2025-05-01 17:13:58.386475: I tensorflow/core/framework/local_rendezvous.cc:428] Local rendezvous send item cancelled. Key hash: 16066635471518802472
2025-05-01 17:13:58.386501: I tensorflow/core/framework/local_rendezvous.cc:428] Local rendezvous send item cancelled. Key hash: 13626566832400366092
2025-05-01 17:13:58.386505: I tensorflow/core/framework/local_rendezvous.cc:428] Local rendezvous send item cancelled. Key hash: 17371444587665041699


95/98 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9279 - loss: 0.2257

2025-05-01 17:14:00.138506: I tensorflow/core/framework/local_rendezvous.cc:428] Local rendezvous send item cancelled. Key hash: 17048839311893398996
2025-05-01 17:14:00.138535: I tensorflow/core/framework/local_rendezvous.cc:428] Local rendezvous send item cancelled. Key hash: 3976896148490793123
2025-05-01 17:14:00.138559: I tensorflow/core/framework/local_rendezvous.cc:428] Local rendezvous send item cancelled. Key hash: 8205433285561994663


98/98 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.9280 - loss: 0.2255 - val_accuracy: 0.8076 - val_loss: 0.7521


2025-05-01 17:14:00.384263: I tensorflow/core/framework/local_rendezvous.cc:428] Local rendezvous send item cancelled. Key hash: 16066635471518802472
2025-05-01 17:14:00.384286: I tensorflow/core/framework/local_rendezvous.cc:428] Local rendezvous send item cancelled. Key hash: 13626566832400366092
2025-05-01 17:14:00.384289: I tensorflow/core/framework/local_rendezvous.cc:428] Local rendezvous send item cancelled. Key hash: 17371444587665041699


In [12]:
model.evaluate(test_set)

34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8022 - loss: 0.7764


2025-05-01 17:14:00.827337: I tensorflow/core/framework/local_rendezvous.cc:428] Local rendezvous send item cancelled. Key hash: 16066635471518802472
2025-05-01 17:14:00.827359: I tensorflow/core/framework/local_rendezvous.cc:428] Local rendezvous send item cancelled. Key hash: 13626566832400366092
2025-05-01 17:14:00.827428: I tensorflow/core/framework/local_rendezvous.cc:428] Local rendezvous send item cancelled. Key hash: 17371444587665041699


[0.7537471055984497, 0.8064346313476562]

In [13]:
import numpy as np
def generate_chorale(model,seed_chords,length):
    arpegio = preprocess(tf.constant(seed_chords,dtype=tf.int64))
    arpegio = tf.reshape(arpegio,[1,-1])
    for chord in range(length):
        for note in range(4):
            next_note = np.argmax(model.predict(arpegio),axis=-1)[:1,-1:]
            arpegio = tf.concat([arpegio,next_note],axis=1)
    
    arpegio = tf.where(arpegio == 0,arpegio,arpegio + min_note -1)
    return tf.reshape(arpegio,shape=[-1,4])

In [14]:
seed_chords = test_chorales[2][:8]
play_chords(seed_chords,amplitude=0.2)

In [15]:
def generate_chorale_v2(mode,seed_chords,length,temperature=1):
    arpegio = preprocess(tf.constant(seed_chords,dtype=tf.int64))
    arpegio = tf.reshape(arpegio,[1,-1])
    for chord in range(length):
        for note in range(4):
            next_note_probas = mode.predict(arpegio)[0,-1:]
            rescaled_logits = tf.math.log(next_note_probas) / temperature
            next_note = tf.random.categorical(rescaled_logits,num_samples=1)
            arpegio = tf.concat([arpegio,next_note],axis=1)

    arpegio = tf.where(arpegio == 0,arpegio,arpegio +min_note - 1)
    return tf.reshape(arpegio,shape=[-1,4])

In [22]:
new_chorale_v2_cold = generate_chorale_v2(model,seed_chords,90,temperature=1.5)
play_chords(new_chorale_v2_cold)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━